# Lab 11: CNN Model for Time-Series Forecasting

**Student Name:** Yousaf Tahir  
**Registration Number:** 22JZELE0479  
**Course:** Machine Learning Lab  
**Supervisor:** Engr. Irshad Ullah  
**University:** UET Nowshera Campus  


## Learning Objectives
- Import TensorFlow/Keras, Scikit-learn metrics, and custom time-series utilities.
- Define a CNN architecture using Conv1D layers for sequential data.
- Configure callbacks, optimizer, checkpoints, and training paths.
- Load train, validation, and test datasets for forecasting.
- Evaluate CNN forecasting performance using standard regression metrics.


## Section 1: Working Directory and Library Imports
This section sets the Lab 10/11 path and imports all libraries required for CNN-based time-series forecasting.


In [2]:
import os
os.chdir(r'D:\Machine Learning\ML Lab\Lab 11')

In [3]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, explained_variance_score, r2_score
from timeseires.utils.to_split import to_split
from timeseires.utils.multivariate_multi_step import multivariate_multi_step
from timeseires.utils.multivariate_single_step import multivariate_single_step
from timeseires.utils.univariate_multi_step import univariate_multi_step
from timeseires.utils.univariate_single_step import univariate_single_step
from timeseires.utils.CosineAnnealingLRS import CosineAnnealingLRS
from timeseires.callbacks.EpochCheckpoint import EpochCheckpoint
from tensorflow.keras.callbacks import ModelCheckpoint
from timeseires.callbacks.TrainingMonitor import TrainingMonitor
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import LSTM, Bidirectional, Add
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import Conv1D,TimeDistributed
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten,MaxPooling1D,Concatenate,AveragePooling1D, GlobalMaxPooling1D, Input
from tensorflow.keras.models import Sequential,Model
import pandas as pd
import time, pickle
import numpy as np
import tensorflow.keras.backend as K
import tensorflow
from tensorflow.keras.layers import Input, Reshape, Lambda
from tensorflow.keras.layers import Layer, Flatten, LeakyReLU, concatenate, Dense
from tensorflow.keras.regularizers import l2
import glob
import h5py
import matplotlib.pyplot as plt
from keras.callbacks import Callback

In [4]:
#lookback = 24
model = None
start_epoch = 0
time_steps=24
num_features=21

In [5]:
def CNN():
    input_data = Input(shape=(time_steps, num_features))
    x1 = Conv1D(16, 2, activation="relu")(input_data)
    x2 = Conv1D(16, 2, activation="relu")(x1)
    flatten = Flatten()(x2)
    output_data = Dense(1)(flatten)
    model = Model(input_data, output_data)
    return model

In [6]:
model1 = CNN()
model1.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 24, 21)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 23, 16)         │           688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 22, 16)         │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 352)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           353 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,569 (6.13 KB)

 Trainable params: 1,569 (6.13 KB)

 Non-trainable params: 0 (0.00 B)

## Section 2: Model Parameters and CNN Architecture
The following cells define input shape parameters and build the Conv1D-based CNN model for forecasting.


In [7]:
tensorflow.keras.utils.plot_model(model1 )

You must install pydot (`pip install pydot`) for `plot_model` to work.


In [8]:
checkpoints = r'D:\Machine Learning\ML Lab\Lab 11\E1-cp-{epoch:04d}-loss{val_loss:.2f}.h5'
OUTPUT_PATH = r'D:\Machine Learning\ML Lab\Lab 11'
FIG_PATH = os.path.sep.join([OUTPUT_PATH,"\history.png"])
JSON_PATH = os.path.sep.join([OUTPUT_PATH,"\history.json"])

In [9]:
# construct the callback to save only the *best* model to disk
# based on the validation loss
EpochCheckpoint1 = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

# construct the set of callbacks
callbacks = [EpochCheckpoint1,TrainingMonitor1]

In [10]:
# if there is no specific model checkpoint supplied, then initialize
# the network and compile the model
if model is None:
    print("[INFO] compiling model...")
    model =CNN()
    opt = Adam(1e-3)
    model.compile(loss= 'mae', optimizer=opt, metrics=["mae", "mape"])
# otherwise, load the checkpoint from disk
else:
    print("[INFO] loading {}...".format(model))
    model = load_model(model)

    # update the learning rate
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.lr)))
    K.set_value(model.optimizer.lr, 1e-4)
    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.lr)))

[INFO] compiling model...


In [11]:
import os
path_dataset =r'D:\Machine Learning\ML Lab\Lab 11'
path_tr = os.path.join(path_dataset, 'train.csv')
df_tr = pd.read_csv(path_tr)
train_set = df_tr.iloc[:].values
path_v = os.path.join(path_dataset, 'validation.csv')
df_v = pd.read_csv(path_v)
validation_set = df_v.iloc[:].values 
path_te = os.path.join(path_dataset, 'test.csv')
df_te = pd.read_csv(path_te)
test_set = df_te.iloc[:].values 

path_scaler = os.path.join(path_dataset, 'AEP_scaler.pkl')
scaler         = pickle.load(open(path_scaler, 'rb'))

train_set.shape, validation_set.shape, test_set.shape

c:\Users\Yousaf Tahir\anaconda3\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.0.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


((860, 21), (90, 21), (30, 21))

## Section 3: Training Configuration and Callback Setup
This section prepares checkpoint saving, monitoring, optimizer configuration, and model compilation for training.


In [12]:
time_steps=24
num_features=21

In [13]:
start = time.time()
train_X , train_y = univariate_multi_step(train_set, time_steps, target_col=0,target_len=1)
validation_X, validation_y = univariate_multi_step(validation_set, time_steps, target_col=0,target_len=1)
test_X, test_y = univariate_multi_step(test_set, time_steps, target_col=0,target_len=1)
print('Time Consumed', time.time()-start, "sec")

Time Consumed 0.003786802291870117 sec


In [14]:
epochs = 10
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,verbose = verbose)

Epoch 1/10
 1/27 ━━━━━━━━━━━━━━━━━━━━ 22s 865ms/step - loss: 0.2960 - mae: 0.2960 - mape: 333.7784
Epoch 1: val_loss improved from None to 0.11112, saving model to D:\Machine Learning\ML Lab\Lab 11\E1-cp-0001-loss0.11.h5



Epoch 1: finished saving model to D:\Machine Learning\ML Lab\Lab 11\E1-cp-0001-loss0.11.h5
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.1401 - mae: 0.1401 - mape: 80.6317 - val_loss: 0.1111 - val_mae: 0.1111 - val_mape: 38.9276
Epoch 2/10
 1/27 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 0.0917 - mae: 0.0917 - mape: 38.3603
Epoch 2: val_loss improved from 0.11112 to 0.07348, saving model to D:\Machine Learning\ML Lab\Lab 11\E1-cp-0002-loss0.07.h5



Epoch 2: finished saving model to D:\Machine Learning\ML Lab\Lab 11\E1-cp-0002-loss0.07.h5
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0736 - mae: 0.0736 - mape: 37.1183 - val_loss: 0.0735 - val_mae: 0.0735 - val_mape: 24.3305
Epoch 3/10
23/27 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0647 - mae: 0.0647 - mape: 35.1903 
Epoch 3: val_loss improved from 0.07348 to 0.05543, saving model to D:\Machine Learning\ML Lab\Lab 11\E1-cp-0003-loss0.06.h5



Epoch 3: finished saving model to D:\Machine Learning\ML Lab\Lab 11\E1-cp-0003-loss0.06.h5
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0646 - mae: 0.0646 - mape: 33.8413 - val_loss: 0.0554 - val_mae: 0.0554 - val_mape: 17.1315
Epoch 4/10
 1/27 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 0.0531 - mae: 0.0531 - mape: 27.5841
Epoch 4: val_loss did not improve from 0.05543
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0558 - mae: 0.0558 - mape: 28.4584 - val_loss: 0.0645 - val_mae: 0.0645 - val_mape: 20.1131
Epoch 5/10
 1/27 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 0.0524 - mae: 0.0524 - mape: 17.3207
Epoch 5: val_loss did not improve from 0.05543
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0544 - mae: 0.0544 - mape: 27.9865 - val_loss: 0.0607 - val_mae: 0.0607 - val_mape: 18.5301
Epoch 6/10
 1/27 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0379 - mae: 0.0379 - mape: 15.3660
Epoch 6: val_loss improved from 0.05543 to 0.04414, saving model to D:\Machine Learning\ML Lab\Lab 11\


Epoch 6: finished saving model to D:\Machine Learning\ML Lab\Lab 11\E1-cp-0006-loss0.04.h5
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0480 - mae: 0.0480 - mape: 25.0501 - val_loss: 0.0441 - val_mae: 0.0441 - val_mape: 14.1585
Epoch 7/10
 1/27 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 0.0516 - mae: 0.0516 - mape: 25.1835
Epoch 7: val_loss improved from 0.04414 to 0.03583, saving model to D:\Machine Learning\ML Lab\Lab 11\E1-cp-0007-loss0.04.h5



Epoch 7: finished saving model to D:\Machine Learning\ML Lab\Lab 11\E1-cp-0007-loss0.04.h5
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0474 - mae: 0.0474 - mape: 24.8303 - val_loss: 0.0358 - val_mae: 0.0358 - val_mape: 11.6345
Epoch 8/10
 1/27 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 0.0544 - mae: 0.0544 - mape: 24.5637
Epoch 8: val_loss did not improve from 0.03583
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0428 - mae: 0.0428 - mape: 22.3335 - val_loss: 0.0438 - val_mae: 0.0438 - val_mape: 13.5205
Epoch 9/10
 1/27 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 0.0518 - mae: 0.0518 - mape: 22.0308
Epoch 9: val_loss did not improve from 0.03583
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0402 - mae: 0.0402 - mape: 20.7246 - val_loss: 0.0614 - val_mae: 0.0614 - val_mape: 20.8947
Epoch 10/10
 1/27 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0504 - mae: 0.0504 - mape: 18.6923
Epoch 10: val_loss did not improve from 0.03583
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0396 

In [15]:

model = load_model(r'D:\Machine Learning\ML Lab\Lab 11\E1-cp-0007-loss0.04.h5', compile=False)

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)
MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squarred Error (RMSE) 
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step
Mean Absolute Error (MAE): 1638.59
Median Absolute Error (MedAE): 1658.82
Mean Squared Error (MSE): 3138565.16
Root Mean Squared Error (RMSE): 1771.6
Mean Absolute Percentage Error (MAPE): 10.46 %
Median Absolute Percentage Error (MDAPE): 10.72 %


y_test_unscaled.shape=  (5, 1)
y_pred.shape=  (5, 1)


In [16]:
checkpoints = r'D:\Machine Learning\ML Lab\Lab 11\E1-cp-0007-loss0.04.h5'
model=r'D:\Machine Learning\ML Lab\Lab 11\E1-cp-0007-loss0.04.h5'
start_epoch= 58

## Section 4: Dataset Loading, Prediction, and Evaluation
The remaining cells load the data, prepare it for the CNN model, generate predictions, and calculate forecasting metrics.


In [17]:
# construct the callback to save only the *best* model to disk
# based on the validation loss
EpochCheckpoint1 = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

# construct the set of callbacks
callbacks = [EpochCheckpoint1,TrainingMonitor1]
model_path = r'D:\Machine Learning\ML Lab\Lab 11\E1-cp-0007-loss0.04.h5'
# if there is no specific model checkpoint supplied, then initialize
# the network and compile the model
if model is None:
    print("[INFO] compiling model...")
    model = PC.build(time_steps=24, num_features=21, reg=0.0005)
    opt = Adam(1e-3)
    model.compile(loss= 'mae', optimizer=opt, metrics=["mae", "mape"])
# otherwise, load the checkpoint from disk
else:
    print("[INFO] loading {}...".format(model_path))
    model = load_model(model_path, compile=False)

    opt = Adam(learning_rate=1e-4)
    model.compile(
        loss="mae",
        optimizer=opt,
        metrics=["mae", "mape"]
    )

    # update the learning rate
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.learning_rate)))
    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.learning_rate)))

[INFO] loading D:\Machine Learning\ML Lab\Lab 11\E1-cp-0007-loss0.04.h5...
[INFO] old learning rate: 9.999999747378752e-05
[INFO] new learning rate: 9.999999747378752e-05


In [18]:
epochs = 10
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,
                        verbose = verbose)

Epoch 1/10


 1/27 ━━━━━━━━━━━━━━━━━━━━ 18s 710ms/step - loss: 0.0413 - mae: 0.0413 - mape: 41.4780
Epoch 1: val_loss improved from None to 0.04066, saving model to D:\Machine Learning\ML Lab\Lab 11\E1-cp-0007-loss0.04.h5



Epoch 1: finished saving model to D:\Machine Learning\ML Lab\Lab 11\E1-cp-0007-loss0.04.h5
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0409 - mae: 0.0409 - mape: 20.8832 - val_loss: 0.0407 - val_mae: 0.0407 - val_mape: 13.5609
Epoch 2/10
 1/27 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0375 - mae: 0.0375 - mape: 20.0464
Epoch 2: val_loss improved from 0.04066 to 0.03947, saving model to D:\Machine Learning\ML Lab\Lab 11\E1-cp-0007-loss0.04.h5



Epoch 2: finished saving model to D:\Machine Learning\ML Lab\Lab 11\E1-cp-0007-loss0.04.h5
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0398 - mae: 0.0398 - mape: 20.1133 - val_loss: 0.0395 - val_mae: 0.0395 - val_mape: 13.0236
Epoch 3/10
 1/27 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 0.0491 - mae: 0.0491 - mape: 29.7032
Epoch 3: val_loss did not improve from 0.03947
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0393 - mae: 0.0393 - mape: 19.8673 - val_loss: 0.0407 - val_mae: 0.0407 - val_mape: 13.5255
Epoch 4/10
 1/27 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0386 - mae: 0.0386 - mape: 15.6954
Epoch 4: val_loss did not improve from 0.03947
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0388 - mae: 0.0388 - mape: 19.4345 - val_loss: 0.0443 - val_mae: 0.0443 - val_mape: 14.8280
Epoch 5/10
 1/27 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 0.0392 - mae: 0.0392 - mape: 16.1246
Epoch 5: val_loss did not improve from 0.03947
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0383 - 

In [19]:

model = load_model(r'D:\Machine Learning\ML Lab\Lab 11\E1-cp-0007-loss0.04.h5', compile=False)

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)
MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squarred Error (RMSE) 
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
Mean Absolute Error (MAE): 1504.02
Median Absolute Error (MedAE): 1526.72
Mean Squared Error (MSE): 2685976.07
Root Mean Squared Error (RMSE): 1638.89
Mean Absolute Percentage Error (MAPE): 9.6 %
Median Absolute Percentage Error (MDAPE): 9.87 %


y_test_unscaled.shape=  (5, 1)
y_pred.shape=  (5, 1)


## Final Conclusion
In this lab, a CNN model was applied to time-series forecasting. The notebook shows how Conv1D layers can extract local temporal patterns and how the trained model is evaluated using regression metrics.